In [9]:
# ----------- Libraries ----------- #
import numpy as np
import pandas as pd
import matplotlib as mpl

# ----------- Constants ----------- #

learning_rate = 0.01


In [ ]:
# ----------- Back Prop Function ----------- #
def back_prop_cost(activations, goal_value, z_vales, weights, biases, output, final_z):

    #some context the activations are your value after the function and your z_values are your before the function

    gradient = np.array([])

    current_activations = activations # saves a clone for calculating the partials for backprop

    new_weights = {}

    new_biases = {}

    layers = len(activations[0])

    for i in range(layers, 0, -1):
        
        c_a = None

        der_relu = None

        numOutputs = 1

        if i == layers: #Special case going back
            c_a = 2 * (output - goal_value)
            der_relu = 1 if final_z > 0 else 0
        else:
            #The conditions and choices for the derivative of our relu function 
            # Note: Since relu does not derive when it equals 0 I just added it to the case of bellow 0
            conditions = [
                (z_vales[:, i - 1] <= 0),
                        (z_vales[:, i - 1] > 0)
                    
            ]
            choices = [0, 1]
            
            der_relu = np.select(conditions, choices)

        if len(weights[i - 1].shape) != 1:
            numOutputs = weights[i - 1].shape[1]
            current_sum = np.transpose(weights[i - 1][0, :])
        else:
            current_sum = np.transpose(weights[i - 1])
        if (len(current_activations.shape) > 1 and c_a == None):
            c_a =  2 * (current_activations[:, i - 1] -  goal_value) #partial of C w respect to A(L)
        elif (c_a == None):
            c_a =  2 * (current_activations[i - 1] -  goal_value) #partial of C w respect to A(L)
    
        
        weights_partial = current_activations[:, i-1] * der_relu * c_a

        bias_partial = der_relu * c_a

        np.append(gradient, weights_partial)
        np.append(gradient, bias_partial)

        #FIX: trying to minimize cost so needs to be negative GD
        new_weights[i - 1] =  -1 *(weights[i - 1] - (learning_rate * weights_partial))
        new_biases[i] = -1 * (biases[i] - (learning_rate * bias_partial))

        #Does the a^L-1 activation function

        if (numOutputs > 1):
            for j in range(1, numOutputs):
                current_sum += weights[i - 1][j, :] * der_relu * c_a

        current_activations[:, i - 1] = current_sum

    return gradient, new_weights, new_biases



In [37]:
# ----------- Forward Prop Functions ----------- #

def identity_matrix(input_weights, weight_layer):
    #Turns this into the equivalent diag matrix
    
    input_shape = input_weights[weight_layer].shape

    #checks if its already in the proper square matrix form
    
    if (len(input_shape) > 1 and input_shape[0] == input_shape[1]):
        return input_weights[weight_layer]

    return np.diag(input_weights[weight_layer])

def layer_processing (input_weights, bias, input_activations, activation_array):
    layers = len(input_activations[0])

    identity_weight = identity_matrix(input_weights, 0)

    for i in range(1,layers):

        layer_biases = bias[i]
        
        vetor_input = np.transpose(input_activations[:, i-1])
        
        vector_output = np.matmul(vetor_input, identity_weight)

        new_vector = vector_output + layer_biases

        activation_array[:, i]  = new_vector # Added so I can get the pre relu function

        new_vector = np.maximum(new_vector, 0)

        input_activations[:, i] =  new_vector 

        if (i != layers - 1):
            identity_weight = identity_matrix(input_weights, i)


    final_act_vector = np.transpose(input_activations[:, layers-1])

    final_weight_vector = input_weights[layers-1]

    final_biased_vector = np.matmul(final_act_vector, final_weight_vector) + bias[layers]

    return np.maximum(final_biased_vector,0), final_biased_vector


biases = {
    1: np.array([2, 
                 3]),
    2: np.array([4,6]),
    3: np.array([-1])
}
vertex_array = np.array([[6, -1, -1],  # example contains an input vector of [6,10] for the middle part of the NN stores our z^Ls
                         [10, -1, -1]
                         ])
activation_array = np.array([[6, -1, -1],  # this is a duplicate array for storing the activiations stuff pre function for later on
                         [10, -1, -1]
                         ])
#each of these need to be transposed to column vectors and they are by layer

#These are the weight outputs for each neuron changed to make back prop easier
edge_output_list = {
    0: np.array([15, 
                 11]),
    1: np.array([[5, 14],
                 [12, 16]]),
    2: np.array([4, 
                 12])
}

weights_log = {}

bias_log = {}

iterations = 1 # however many iterations of backprop that there are

goal_output = 50000

max_iteration = 10

while True and iterations <= max_iteration:
    
    fp, final_z = layer_processing(edge_output_list, biases, vertex_array, activation_array)

    difference = np.abs(goal_output - fp)

    if difference < 1:
        break

    print(f"Iteration: {iterations} difference {difference}")
    gradient, new_weights, new_biases = back_prop_cost(vertex_array, goal_output, activation_array, edge_output_list, biases, fp, final_z)

    weights_log[iterations] = new_weights

    bias_log[iterations] = new_biases

    print(f"New Weights: {new_weights}")

    print(f"New Biases: {new_biases}")

    #TODO: REMOVE HARDCODED IMPLEMENTATION FOR RESETTING THE MATRIX BACK
    vertex_array = np.array([[6, -1, -1], 
                         [10, -1, -1]
                         ])

    activation_array = np.array([[6, -1, -1],  # this is a duplicate array for storing the activations stuff pre function for later on
                         [10, -1, -1]
                         ])

    

    iterations += 1

print(f"Learning cycle for the training input completed after {iterations} iteration(s) ")



Iteration: 1 difference [5497]
New Weights: {2: array([200094.8 , 341045.88]), 1: array([[ 91835.72, 112758.62],
       [ 91842.72, 112760.62]]), 0: array([ 6014.28, 10009.  ])}
New Biases: {3: array([108.94]), 2: array([1002.16, 1003.74]), 1: array([1001.88, 1002.8 ])}
Iteration: 2 difference [50000]
New Weights: {2: array([ 4., 12.]), 1: array([[-1105956.28, -1483625.38],
       [   91842.72,   112760.62]]), 0: array([ 6014.28, 10009.  ])}
New Biases: {3: array([-1.]), 2: array([1002.16, 1003.74]), 1: array([1001.88, 1002.8 ])}
Iteration: 3 difference [50000]
New Weights: {2: array([ 4., 12.]), 1: array([[-2303748.28, -3080009.38],
       [   91842.72,   112760.62]]), 0: array([ 6014.28, 10009.  ])}
New Biases: {3: array([-1.]), 2: array([1002.16, 1003.74]), 1: array([1001.88, 1002.8 ])}
Iteration: 4 difference [50000]
New Weights: {2: array([ 4., 12.]), 1: array([[-3501540.28, -4676393.38],
       [   91842.72,   112760.62]]), 0: array([ 6014.28, 10009.  ])}
New Biases: {3: array([-